In [ ]:
import os
import pandas as pd
from collections import Counter


# 데이터 로드

In [ ]:
# 형태소 처리된 비공개 분석 입력 위치
file_path = os.environ['LG_VOC_DATA_DIR'] + os.sep

styler = pd.read_csv(file_path + '의류관리기_수정.csv', encoding='utf-8-sig')
vacuum = pd.read_csv(file_path + '무선스틱청소기_수정.csv', encoding='utf-8-sig')


## 불용어 처리

In [ ]:
### 불용어 처리 리스트
stop_words = {'이다', '에서', '후', '번', '일', '랑', '보다', '를', '와', '내', '과', '되다', '전자서비스센터',
              '시', '때', '데', '을', '는', '가', '이', '개', '때문', '꼭', '경우', '은', '에도',
              '되어다', '수', '의', '없다', '나', '까지', '이제', '같다', '뿐', '해주다', '않다',
              '다이렉트결혼준비', '에', '전', '들', '중', '다', '기', '그렇다', '니', '도', '등', '있다',
              '분', '더', '정말', '습', '게', '하다', '적', '아니다', '매', '더욱', '슈', '인', '안', '해', 'S',
              '만', '바로', '로', '한', 'CB', '처럼', '니다', '그', '오브', '그리고', '통해', '점', '제', '이나',
              '한번', '글', '체험단', '요', '으로', '저희', '거', '하고', '것', '부터', '에는', '기다', '저',
              '블랙틴트미러', '그래서', '특히', '예요', '또한', '많이', '없이', '호텔', '수업', '교시', '출판',
              '서점', '이벤트', '발매', '돼다', '보시', '이라', '참고', '오다', '맞다', '같이', '근데', '그냥', '약간',
              '다음', '여기', '조금', '수가', '하나', '대해', '어떻다', '싶다', '이라고', '너무', '정도', '자주', '이렇게',
              '인데', '이기', '가장', '하지만', '해당', '한다는', '사하', '가장', '는걸', '아주', '고', '\n', '\xa0',
              ' ', '레몬테라스카페', '즉', '다음', '동영상', '취소', '재생', '속도', '음소거', '음소거설정', '실시간', '설정',
              '실시간설정', '다시시도',
              '골프', '레슨', '도움말', '도움말상태', '라이센스', '고화질', '영상', '해상도', '문제', '발생',
              '해상도', '자막', '안함', '초', '광고', '공유', '전체화면', '자막안함옵션',
              '글자', '크기배경색', '도움말', '상태', '배경색기본', '골프연습장', '회원님',
              '골프레슨', '죄송하다', '문제발생', '가능한영상', '자막안함', '글자크기',
              '범례부동산거리', '전체화면', '크기배경색','옵션', '기본자동옵션', '시가총액', '억원', '상승',
              '차량용', '차량', '용', '원', '현재', '동시', '군', '구시', '거리', '읍', '면', '도국', '지도',
              '범례', '부동산', '거리', '데이터', '다시시도', '존', '인', '피트', '회원님', '들다', '기본옵션', '오후', '우리집',
              '년', '집', '및', '준비물', '다시', '시도', '로켓', '배송'
              }

In [ ]:
def remove_stopwords_from_df(df, columns, stop_words):
    # 불용어 처리 함수
    def remove_stopwords(text, stop_words):
        if not isinstance(text, str):
            return ''
        
        # 띄어쓰기를 기준으로 단어 분리
        tokens = text.split()
        
        # 불용어 제거
        filtered_tokens = [word for word in tokens if word not in stop_words]
        
        # 단어를 다시 문장으로 합치기
        return ' '.join(filtered_tokens)

    # 지정된 열에 대해 불용어 처리
    for col in columns:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: remove_stopwords(x, stop_words))
    
    return df

styler = remove_stopwords_from_df(styler, ['noun', 'verb_adj', 'nva', 'token'], stop_words)
vacuum = remove_stopwords_from_df(vacuum, ['noun', 'verb_adj', 'nva', 'token'], stop_words)


## 연도/월별 열 추가

In [ ]:
# year와 month 열 추가 함수
def add_year_month_columns(df):
    df['year'] = pd.to_datetime(df['date']).dt.year
    df['month'] = pd.to_datetime(df['date']).dt.month

# 각 데이터프레임에 함수 적용
add_year_month_columns(styler)
add_year_month_columns(vacuum)


# 핵심 키워드 선정: 감정어 리스트 빈도분석/연도별

* 특정 감정어 (예: 어렵다)가 자주 등장

* 특정 감정어 (예: 불편)가 연도별 증가

* 해당 감정어 핵심 키워드 선정

    1. 연관 키워드 연도별 빈도 변화 추이 보기
    
    2. 연도별로 유의미하게 변화한 키워드랑 연결해서 설명

## 감정어

In [ ]:
emotion_words = [# '사용하다', 
                 '편하다', '편리하다', '만족스럽다', '만족하다', '유용하다', 
                 '간편하다', '좋다', '재미있다', '재밌다', '즐겁다', '흥미롭다', '놀랍다', 
                 '기대하다', '기쁘다', '신기하다', '설레다', '새롭다', '후회없다',
                 
                 '불편하다', '불만족스럽다', '어렵다', '답답하다', '짜증나다', '피곤하다', 
                 '부담스럽다', '부담되다', '귀찮다', '아쉽다', '실망하다', '싫다', '열받다', 
                 '후회하다', '실망하다', '화나다', '빡치다', '구리다', '어렵다'
                 ]

### 전체

In [ ]:
import pandas as pd
from collections import Counter

def count_emotions(text_series, words, year_series=None, year1=None, year2=None):
    # NaN 제거 및 문자열 변환
    valid_texts = text_series.dropna().astype(str)
    all_text = ' '.join(valid_texts)  # 모든 텍스트를 하나로 합침

    # 단어 빈도 계산
    word_counts = Counter([word for word in all_text.split() if word in words])
    total_word_count = sum(word_counts.values())

    # 상위 단어 추출 및 비율 계산
    most_common_words = word_counts.most_common()
    words, freqs = zip(*most_common_words) if most_common_words else ([], [])
    ratios = [freq / total_word_count for freq in freqs]

    # 연도별 변화율(transition) 계산
    transitions = []
    if year_series is not None and year1 is not None and year2 is not None:
        # 연도별 데이터 필터링
        year1_texts = text_series[year_series == year1].dropna().astype(str)
        year2_texts = text_series[year_series == year2].dropna().astype(str)

        year1_words = ' '.join(year1_texts).split()
        year2_words = ' '.join(year2_texts).split()

        year1_counts = Counter([word for word in year1_words if word in words])
        year2_counts = Counter([word for word in year2_words if word in words])

        total1 = sum(year1_counts.values()) or 1  # 분모 0 방지
        total2 = sum(year2_counts.values()) or 1

        for word in words:
            count1 = year1_counts.get(word, 0)
            count2 = year2_counts.get(word, 0)
            ratio1 = count1 / total1
            ratio2 = count2 / total2
            transition_rate = (ratio2 - ratio1) / ratio1 * 100 if ratio1 > 0 else 0
            transitions.append(transition_rate)
    else:
        transitions = [0] * len(words)  # 연도별 데이터가 없는 경우 0으로 채움

    # 결과 데이터프레임 생성
    result_df = pd.DataFrame({
        'word': words,
        'freq': freqs,
        'ratio': ratios,
        'transition': transitions
    })

    # 빈도(freq) 기준으로 순위 계산 및 rank 열 추가
    result_df['rank'] = result_df['freq'].rank(method='dense', ascending=False).astype(int)

    # rank 열을 가장 왼쪽으로 이동
    result_df = result_df[['rank', 'word', 'freq', 'ratio', 'transition']]

    return result_df


In [ ]:
styler_emotion_counts = count_emotions(text_series=styler['nva'], words=emotion_words, year_series=styler['year'], year1=2022, year2=2024)
vacuum_emotion_counts = count_emotions(text_series=vacuum['nva'], words=emotion_words, year_series=vacuum['year'], year1=2022, year2=2024)

### 연도별: 2022 ~ 2024

In [ ]:
import pandas as pd
from collections import Counter

def count_emotions_by_year(text_series, year_series, words, year_range):
    # 유효한 데이터로 필터링
    valid_data = pd.DataFrame({'text': text_series, 'year': year_series}).dropna()
    valid_data['text'] = valid_data['text'].astype(str)

    # 연도 범위 필터링
    valid_data = valid_data[valid_data['year'].isin(year_range)]
    unique_years = sorted(valid_data['year'].unique())

    # 결과 저장용 딕셔너리 초기화
    result_dict = {}

    # 이전 연도 단어 빈도 초기화
    prev_year_counts = Counter()

    for i, year in enumerate(unique_years):
        # 현재 연도의 텍스트 데이터
        current_texts = valid_data[valid_data['year'] == year]['text']
        all_text = ' '.join(current_texts)

        # 단어 빈도 계산
        word_counts = Counter([word for word in all_text.split() if word in words])
        total_word_count = sum(word_counts.values()) or 1  # 분모 0 방지

        # 단어 비율 계산
        words_list, freqs = zip(*word_counts.most_common()) if word_counts else ([], [])
        ratios = [freq / total_word_count for freq in freqs]

        # 이전 연도 대비 변화율(transition) 계산
        transitions = []
        if i == 0:  # 첫 번째 연도는 변화율 계산하지 않음
            transitions = [0] * len(words_list)
        else:
            for word in words_list:
                prev_freq = prev_year_counts.get(word, 0)
                prev_total = sum(prev_year_counts.values()) or 1
                current_ratio = word_counts[word] / total_word_count
                prev_ratio = prev_freq / prev_total

                # 변화율 계산
                transition = (current_ratio - prev_ratio) / prev_ratio * 100 if prev_ratio > 0 else 0
                transitions.append(transition)

        # 현재 연도의 결과를 딕셔너리에 저장
        result_dict[f'word_{year}'] = words_list
        result_dict[f'freq_{year}'] = freqs
        result_dict[f'ratio_{year}'] = ratios
        result_dict[f'transition_{year}'] = transitions

        # 현재 연도의 단어 빈도를 이전 연도로 설정
        prev_year_counts = word_counts

    # 연도별 결과를 같은 감정어 기준으로 정렬
    all_words = sorted(set().union(*[set(result_dict[f'word_{year}']) for year in unique_years]))
    final_result = pd.DataFrame({'word': all_words})
    for year in unique_years:
        year_data = pd.DataFrame({
            'word': result_dict[f'word_{year}'],
            f'freq_{year}': result_dict[f'freq_{year}'],
            f'ratio_{year}': result_dict[f'ratio_{year}'],
            f'transition_{year}': result_dict[f'transition_{year}']
        })
        final_result = final_result.merge(year_data, on='word', how='left')
        final_result[f'word_{year}'] = final_result['word'].where(final_result[f'freq_{year}'].notna())

    # 전체 기간 빈도 기준 순위
    freq_cols = [f'freq_{year}' for year in unique_years]
    final_result.insert(0, 'rank', final_result[freq_cols].fillna(0).sum(axis=1).rank(method='dense', ascending=False).astype(int))
    return final_result.sort_values('rank').reset_index(drop=True)


In [ ]:
# 기간 설정
year_range = [2022, 2023, 2024]

styler_emotion_counts_years = count_emotions_by_year(text_series=styler['nva'], year_series=styler['year'], words=emotion_words, year_range=year_range)
vacuum_emotion_counts_years = count_emotions_by_year(text_series=vacuum['nva'], year_series=vacuum['year'], words=emotion_words, year_range=year_range)


vacuum_emotion_counts_years 

### csv 저장

In [ ]:
# CSV로 저장하기
emotion_counts_path = os.path.join("results", "감정어분석") + os.sep
os.makedirs(emotion_counts_path, exist_ok=True)

styler_emotion_counts.to_csv(emotion_counts_path + 'styler_emotion_counts.csv', index = False, encoding='utf-8-sig')
vacuum_emotion_counts.to_csv(emotion_counts_path + 'vacuum_emotion_counts.csv', index = False, encoding='utf-8-sig')
styler_emotion_counts_years.to_csv(emotion_counts_path + 'styler_emotion_counts_years.csv', index=False, encoding="utf-8-sig")
vacuum_emotion_counts_years.to_csv(emotion_counts_path + 'vacuum_emotion_counts_years.csv', index=False, encoding="utf-8-sig")

## 연관 키워드 연도별 빈도 변화 추이

### 연도별

In [ ]:
import pandas as pd

# 연관 단어 함수
def coocc_word(sentences, keyword, window=10):
    coocc = dict()  # 단어 빈도를 저장할 사전
    for sent in sentences:
        word_list = sent.split(' ')  # 문장을 단어 리스트로 분리
        length = len(word_list)
        for i, word in enumerate(word_list):
            if word == keyword:  # 현재 단어와 키워드가 일치하면,
                for j in range(max(i - window, 0), min(i + window + 1, length)):  # 주변 단어 탐색
                    if word_list[j] in coocc:
                        coocc[word_list[j]] += 1
                    else:
                        coocc[word_list[j]] = 1
    df_occ = pd.Series(coocc, name='occ').sort_values(ascending=False)
    
    # keyword를 포함한 단어 제거
    df_occ = df_occ[df_occ.index != keyword]
    
    return df_occ

def generate_column_order(years):
    base_columns = ['rank']
    
    for year in years:
        base_columns.append(f'word_{year}')
        base_columns.append(f'freq_{year}')
        base_columns.append(f'ratio_{year}')
        base_columns.append(f'transition_{year}')
    
    return base_columns

# 연관 단어 결과로 rank, word, freq, ratio 데이터프레임 생성 함수 (transition 수정)
def coocc_word_with_rank(text_series, year_series, keyword, years, window=10, top_n=None, path=None, df_name=None):
    # 유효한 데이터로 필터링
    valid_data = pd.DataFrame({'text': text_series, 'year': year_series}).dropna()
    valid_data['text'] = valid_data['text'].astype(str)

    # 결과 저장용 딕셔너리 초기화
    result_dict = {}

    prev_year_counts = {}  # 이전 연도의 단어 빈도를 저장할 딕셔너리

    for i, year in enumerate(years):
        # 현재 연도의 텍스트 데이터
        current_texts = valid_data[valid_data['year'] == year]['text']
        # 게시글 사이의 단어가 연관어로 집계되지 않도록 행별로 계산
        sentences = current_texts.dropna().astype(str).tolist()
        df_occ = coocc_word(sentences, keyword, window)

        # 단어 빈도 계산
        word_counts = df_occ.to_dict()  # 현재 연도 단어 빈도를 딕셔너리로 저장
        words_list = list(word_counts.keys())
        freqs = list(word_counts.values())

        total_word_count = sum(freqs) or 1  # 분모 0 방지

        # 단어 비율 계산
        ratios = [freq / total_word_count for freq in freqs]

        # transition 계산 (이전 연도 대비 변화율)
        transitions = []
        if i == 0:  # 첫 번째 연도는 변화율 계산하지 않음
            transitions = [0] * len(words_list)
        else:
            for word in words_list:
                prev_freq = prev_year_counts.get(word, 0)
                prev_total = sum(prev_year_counts.values()) or 1
                current_ratio = word_counts[word] / total_word_count
                prev_ratio = prev_freq / prev_total

                # 변화율 계산
                transition = (current_ratio - prev_ratio) / prev_ratio * 100 if prev_ratio > 0 else current_ratio * 100
                transitions.append(transition)

        # 현재 연도의 결과를 딕셔너리에 저장
        result_dict[f'word_{year}'] = words_list
        result_dict[f'freq_{year}'] = freqs
        result_dict[f'ratio_{year}'] = ratios
        result_dict[f'transition_{year}'] = transitions

        # 현재 연도의 단어 빈도를 이전 연도로 저장
        prev_year_counts = word_counts

    # 연도별 결과를 같은 연관 단어 기준으로 정렬
    all_words = sorted(set().union(*[set(result_dict[f'word_{year}']) for year in years]))
    final_result = pd.DataFrame({'word': all_words})
    for year in years:
        year_data = pd.DataFrame({
            'word': result_dict[f'word_{year}'],
            f'freq_{year}': result_dict[f'freq_{year}'],
            f'ratio_{year}': result_dict[f'ratio_{year}'],
            f'transition_{year}': result_dict[f'transition_{year}']
        })
        final_result = final_result.merge(year_data, on='word', how='left')
        final_result[f'word_{year}'] = final_result['word'].where(final_result[f'freq_{year}'].notna())

    # 전체 기간 빈도 기준 순위
    freq_cols = [f'freq_{year}' for year in years]
    final_result.insert(0, 'rank', final_result[freq_cols].fillna(0).sum(axis=1).rank(method='dense', ascending=False).astype(int))
    final_result = final_result.sort_values('rank').reset_index(drop=True)

    # top_n이 지정되면 상위 n개의 결과만 선택
    if top_n:
        final_result = final_result.head(top_n)

    # 결과를 CSV로 저장
    if path:
        os.makedirs(path, exist_ok=True)
        column_name = text_series.name or 'text'
        filename = f"{df_name}_{column_name}_{keyword}.csv"
        filepath = os.path.join(path, filename)
        final_result.to_csv(filepath, index=False, encoding="utf-8-sig")
        print(f"Results saved to {filepath}")

    return final_result


### 의류관리기 키워드

- **편리하다**
- **간편하다**
- **불편하다**
- **놀랍다**
- **후회없다**
- **흥미롭다**

- **어렵다**
- **새롭다**  
- **만족스럽다**
- **유용하다**
- **만족하다**
- **부담스럽다**
- **짜증나다**
- **구리다**
- **후회하다**
- **피곤하다**
- **편하다**
- **귀찮다**

In [ ]:
# 경로
coocc_emotion_counts_path = os.path.join('results', '감정어분석', '연관키워드') + os.sep
os.makedirs(coocc_emotion_counts_path, exist_ok=True)

# 2022, 2023, 2024년을 처리
years = [2022, 2023, 2024]


# 키워드
# styler_nav_어렵다.csv 저장됨
coocc_word_with_rank(styler['nva'], styler['year'], keyword='귀찮다', years=[2022, 2023, 2024], 
                     top_n=21, path=coocc_emotion_counts_path, df_name= 'styler')



### 무선스틱청소기 키워드

- **편리하다**
- **간편하다**
- **유용하다**
- **만족하다**
- **어렵다**
- **아쉽다**
- **후회하다**
- **후회없다**
- **놀랍다**
- **재미있다**
- **실망하다**
- **즐겁다**
- **빡치다**

- **만족스럽다**
- **불편하다**
- **기대하다**
- **부담스럽다**
- **짜증나다**
- **피곤하다**
- **설레다**
- **신기하다**
- **새롭다**

In [ ]:
# 경로
coocc_emotion_counts_path = os.path.join('results', '감정어분석', '연관키워드') + os.sep
os.makedirs(coocc_emotion_counts_path, exist_ok=True)

# 2022, 2023, 2024년을 처리
years = [2022, 2023, 2024]


# 키워드
# vacuum_nva_편리하다.csv 저장됨
coocc_word_with_rank(vacuum['nva'], vacuum['year'], keyword='편리하다', years=[2022, 2023, 2024], 
                     top_n=21, window = 1, path=coocc_emotion_counts_path, df_name= 'vacuum')


# 증감 check!

## 함수

In [ ]:
# 1. 2022 -> 2023 -> 2024 transition 값이 모두 증가한 단어를 확인
def consistently_increasing_words(df):
    increasing_words = []
    
    # 각 단어 확인
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 확인
        if word in df['word_2023'].values:
            transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
            
            # transition_2023 값이 양수인 경우에만 계속 진행
            if transition_2023 > 0:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 양수인 경우 단어 추가
                    if transition_2024 > 0:
                        increasing_words.append(word)
    
    return increasing_words


# 1. 2022 -> 2023 -> 2024 transition 값이 모두 감소한 단어를 확인
def consistently_decreasing_words(df):
    decreasing_words = []
    
    # 각 단어 확인
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 확인
        if word in df['word_2023'].values:
            transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
            
            # transition_2023 값이 음수인 경우에만 계속 진행
            if transition_2023 < 0:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 음수인 경우 단어 추가
                    if transition_2024 < 0:
                        decreasing_words.append(word)
    
    return decreasing_words


# 1. 2022 -> 2023에는 감소, 2023 -> 2024에는 transition 값이 증가한 단어를 확인
# 2. 2022에는 존재하는 단어지만 2023에는 사라진 단어. 2024에 다시 등장 & transition 값이 증가(양수)한 단어
def recently_increasing_words(df):
    recently_increasing_words = []
    
    # 각 단어 확인
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 확인
        if word in df['word_2023'].values:
            transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
            
            # transition_2023 값이 음수인 경우에만 계속 진행
            if transition_2023 < 0:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 양수인 경우 단어 추가
                    if transition_2024 > 0:
                        recently_increasing_words.append(word)
    
    # 조건 2
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 없는거 확인
        if word not in df['word_2023'].values:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 양수인 경우 단어 추가
                    if transition_2024 > 0:
                        recently_increasing_words.append(word)

    # 조건 3
    for word in df['word_2023']:
        if pd.isna(word):
            continue
        # transition_2023 값이 양수인 경우에만 계속 진행
        transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
        if transition_2023 > 0:
            # 2022 데이터에서 해당 단어 없는거 확인
            if word not in df['word_2022'].values:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 양수인 경우 단어 추가
                    if transition_2024 > 0:
                        recently_increasing_words.append(word)

    return recently_increasing_words


# 1. 2022 -> 2023에는 증가, 2023 -> 2024에는 transition 값이 감소한 단어를 확인
def recently_decreasing_words(df):
    recently_decreasing_words = []
    
    # 각 단어 확인
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 확인
        if word in df['word_2023'].values:
            transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
            
            # transition_2023 값이 양수인 경우에만 계속 진행
            if transition_2023 > 0:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 음수인 경우 단어 추가
                    if transition_2024 < 0:
                        recently_decreasing_words.append(word)

    # 조건 2
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 없는거 확인
        if word not in df['word_2023'].values:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 음수인 경우 단어 추가
                    if transition_2024 < 0:
                        recently_decreasing_words.append(word)

    # 조건 3
    for word in df['word_2023']:
        if pd.isna(word):
            continue
        # transition_2023 값이 음수인 경우에만 계속 진행
        transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
        if transition_2023 < 0:
            # 2022 데이터에서 해당 단어 없는거 확인
            if word not in df['word_2022'].values:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 음수인 경우 단어 추가
                    if transition_2024 < 0:
                        recently_decreasing_words.append(word)

    return recently_decreasing_words


# 1. 2024년 상위 목록에 새로 등장한 단어를 확인
def recently_appeared_words(df):
    recently_appeared_words = []
                          
    # word_2022, word_2023에는 없지만 word_2024에만 존재하는 단어 확인
    for word in df['word_2024']:
        if pd.isna(word):
            continue
        if word not in df['word_2022'].values and word not in df['word_2023'].values:
            recently_appeared_words.append(word)


    return recently_appeared_words



## 실행

In [ ]:
df = coocc_word_with_rank(vacuum['nva'], vacuum['year'], keyword='편리하다', years=[2022, 2023, 2024], 
                          top_n=21, window= 1)

print("2022 -> 2023 -> 2024 transition 값이 모두 증가한 단어:")
print(consistently_increasing_words(df))
print("2022 -> 2023 -> 2024 transition 값이 모두 감소한 단어:")
print(consistently_decreasing_words(df))
print("2023 -> 2024 transition 값이 증가한 단어:")
print(recently_increasing_words(df))
print("2023 -> 2024 transition 값이 감소한 단어:")
print(recently_decreasing_words(df))
print("2024년 상위 목록에 새로 등장한 단어:")
print(recently_appeared_words(df))

In [ ]:
df = coocc_word_with_rank(styler['nva'], styler['year'], keyword='제품', years=[2022, 2023, 2024], 
                     top_n=21, window= 10)


print("2022 -> 2023 -> 2024 transition 값이 모두 증가한 단어:")
print(consistently_increasing_words(df))
print("2022 -> 2023 -> 2024 transition 값이 모두 감소한 단어:")
print(consistently_decreasing_words(df))
print("2023 -> 2024 transition 값이 증가한 단어:")
print(recently_increasing_words(df))
print("2023 -> 2024 transition 값이 감소한 단어:")
print(recently_decreasing_words(df))
print("2024년 상위 목록에 새로 등장한 단어:")
print(recently_appeared_words(df))